# Day 3 - Code breaks, block 1: the training loop and its knobs

Short code breaks that go with the slides (3-5 minutes each):
1. **run** the cell and look at the output,
2. **change** the number marked with `# 👈`, run again,
3. open **👀 What you should see** to check.

A **CPU runtime is enough**. You cannot break anything - if something looks strange: *Runtime → Restart session*,
then run the first two cells again.

**Data:** the *Iris* flowers - 150 flowers, 4 measurements each, 3 kinds of iris.

## Setup - run these two cells first
The first cell loads the flowers: 100 for training, 50 for validation.

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

iris = load_iris()
X_train, X_val, y_train, y_val = train_test_split(
    iris.data, iris.target, test_size=50, stratify=iris.target, random_state=0)

# standardise the 4 measurements (like day 1) - mean and spread of the TRAINING flowers only
mean, std = X_train.mean(0), X_train.std(0)
X_train = torch.tensor((X_train - mean) / std, dtype=torch.float32)
X_val = torch.tensor((X_val - mean) / std, dtype=torch.float32)
y_train = torch.tensor(y_train)
y_val = torch.tensor(y_val)

print("training flowers:", len(X_train), "  validation flowers:", len(X_val))

The second cell defines three small helpers. **Just run it - you do not need to read it.**
- `make_model()` builds a small network (4 inputs → 16 hidden neurons → 3 outputs)
- `train(...)` is the training loop from code break 1; it returns the training loss of every epoch
- `accuracy(...)` = share of correct predictions
- `few_virginica(...)` and `virginica_found(...)` are only for code break 3

In [ ]:
from torch.utils.data import DataLoader, TensorDataset


def make_model(hidden=16, n_out=3):
    torch.manual_seed(0)  # same start weights for everybody
    return nn.Sequential(nn.Linear(4, hidden), nn.ReLU(), nn.Linear(hidden, n_out))


def train(model, opt, X=None, y=None, loss_fn=None, epochs=30, batch_size=16):
    X = X_train if X is None else X
    y = y_train if y is None else y
    loss_fn = nn.CrossEntropyLoss() if loss_fn is None else loss_fn
    torch.manual_seed(0)  # same batch order for every run -> fair comparison
    loader = DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=True)
    losses = []
    for epoch in range(epochs):
        for xb, yb in loader:
            loss = loss_fn(model(xb), yb)
            opt.zero_grad()
            loss.backward()
            opt.step()
        with torch.no_grad():
            losses.append(loss_fn(model(X), y).item())
    return losses


def accuracy(model, X, y):
    with torch.no_grad():
        return round((model(X).argmax(1) == y).float().mean().item(), 2)


def few_virginica(X, y, n_keep):
    is_virg = (y == 2).long()                       # task: virginica yes (1) / no (0)
    keep = (is_virg == 0) | (torch.cumsum(is_virg, 0) <= n_keep)
    return X[keep], is_virg[keep]


def virginica_found(model, X, y):
    with torch.no_grad():
        pred = model(X).argmax(1)
    return int(((pred == 1) & (y == 1)).sum())


print("helpers ready")

## 💻 Code break 1 - The training loop: seven knobs
The loop from the slide. The knobs are the numbers at the top.

In [ ]:
HIDDEN = 16        # ① model: number of hidden neurons   👈
LR = 0.01          # ④ learning rate                      👈
BATCH_SIZE = 16    # ⑥ batch size                         👈
EPOCHS = 30        # ⑦ epochs                             👈

model = make_model(hidden=HIDDEN)                                     # ① model
loss_fn = nn.CrossEntropyLoss()                                       # ② loss
opt = torch.optim.AdamW(model.parameters(), lr=LR)                    # ③ optimiser, ④ learning rate
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS) # ⑤ LR schedule
loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)  # ⑥

for epoch in range(EPOCHS):                                           # ⑦ epochs
    for x, y in loader:
        loss = loss_fn(model(x), y)   # forward: how wrong are we?
        opt.zero_grad()
        loss.backward()               # backward: gradients (day 2)
        opt.step()                    # update: change the weights
    sched.step()

print("last batch loss:", round(loss.item(), 3))
print("validation accuracy:", accuracy(model, X_val, y_val))

🔧 **Change it** (one at a time, then set it back):
1. `EPOCHS = 5` - is the model still good?
2. `BATCH_SIZE = 100` - all training flowers in one batch. Better or worse after 30 epochs?
3. `HIDDEN = 2` - a tiny model. What happens?

<details><summary>👀 What you should see</summary>

- original: validation accuracy **0.96**
- 5 epochs: **0.80** - not enough training
- batch size 100: **0.82** - only *one* step per epoch instead of 7, so far fewer steps in total
- 2 hidden neurons: **0.86** - the model is too small
</details>

## 💻 Code break 2 - Give the loss the raw scores (logits)
Two identical networks. Network B has an extra softmax at the end - the bug from the slide.

In [ ]:
model_a = make_model()                                   # correct: ends with raw scores
model_b = nn.Sequential(make_model(), nn.Softmax(dim=1)) # bug: extra softmax at the end

loss_a = train(model_a, torch.optim.AdamW(model_a.parameters(), lr=0.01))
loss_b = train(model_b, torch.optim.AdamW(model_b.parameters(), lr=0.01))

plt.plot(loss_a, label="A: correct")
plt.plot(loss_b, label="B: extra softmax (bug)")
plt.xlabel("epoch"); plt.ylabel("training loss"); plt.legend(); plt.show()
print("final loss  A:", round(loss_a[-1], 2), "  B:", round(loss_b[-1], 2))

🔧 **Change it:**
1. Train longer: add `epochs=100` inside both `train(...)` calls. Does B catch up?
2. Print `accuracy(model_b, X_val, y_val)`. Does the accuracy show the bug?

<details><summary>👀 What you should see</summary>

- A ends at a loss of **0.07**, B is stuck at **0.59** - it can never go below 0.55 (slide "Pitfall")
- 100 epochs: A **0.05**, B still **0.57**
- the accuracy of B looks fine (**0.98**) - **no error message**. That is why this bug is dangerous.
</details>

## 💻 Code break 3 - Class imbalance: weight the loss
New task: "is this flower a *virginica*?" (1 = yes, 0 = no). We keep only **4 virginica** for training -
rare, like seizure windows.

In [ ]:
X_rare, y_rare = few_virginica(X_train, y_train, n_keep=4)
X_rare_val, y_rare_val = few_virginica(X_val, y_val, n_keep=5)
print("training flowers:", len(y_rare), "  virginica among them:", int(y_rare.sum()))
print("validation flowers:", len(y_rare_val), "  virginica among them:", int(y_rare_val.sum()))

Now the experiment. `WEIGHT` = how much one virginica counts in the loss (1.0 = like every other flower).

In [ ]:
WEIGHT = 1.0       # 👈 try 3.0, then 16.5

loss_fn = nn.CrossEntropyLoss(weight=torch.tensor([1.0, WEIGHT]))   # ② loss with class weights
model = make_model(n_out=2)
train(model, torch.optim.AdamW(model.parameters(), lr=0.01), X_rare, y_rare, loss_fn, epochs=200, batch_size=100)

print("accuracy:", accuracy(model, X_rare_val, y_rare_val))
print("virginica found:", virginica_found(model, X_rare_val, y_rare_val), "of 5")

🔧 **Change it:**
1. `WEIGHT = 3.0`, then `WEIGHT = 16.5` (there are 66 other flowers and 4 virginica: 66 / 4 = 16.5).
2. Which number would you report to a doctor - the accuracy or "found ... of 5"?

<details><summary>👀 What you should see</summary>

- weight 1: accuracy **0.92** - looks good! - but only **2 of 5** virginica found
  (a model that always says "no" would already get 34 / 39 = 0.87)
- weight 3: **3 of 5** (accuracy 0.92)
- weight 16.5: **5 of 5**, accuracy **0.97**
- accuracy hides the missed class - more about this in block 3
</details>

## 💻 Code break 4 - Optimisers: SGD, momentum, Adam
Same network, same data, same learning rate - only knob ③ changes.

In [ ]:
MOMENTUM = 0.9     # 👈

model_1 = make_model()
loss_sgd = train(model_1, torch.optim.SGD(model_1.parameters(), lr=0.01))

model_2 = make_model()
loss_momentum = train(model_2, torch.optim.SGD(model_2.parameters(), lr=0.01, momentum=MOMENTUM))

model_3 = make_model()
loss_adam = train(model_3, torch.optim.Adam(model_3.parameters(), lr=0.01))

plt.plot(loss_sgd, label="SGD")
plt.plot(loss_momentum, label="SGD + momentum")
plt.plot(loss_adam, label="Adam")
plt.xlabel("epoch"); plt.ylabel("training loss"); plt.legend(); plt.show()
print("final loss  SGD:", round(loss_sgd[-1], 2), "  momentum:", round(loss_momentum[-1], 2),
      "  Adam:", round(loss_adam[-1], 2))

🔧 **Change it:**
1. `MOMENTUM = 0.5` - where does the orange curve end now?
2. `MOMENTUM = 0.99` - a very heavy ball. Better or worse here?

<details><summary>👀 What you should see</summary>

- SGD is slow (**0.57**), momentum is much faster (**0.14**), Adam is fastest (**0.07**)
- momentum 0.5: **0.39**
- momentum 0.99: **0.05** - on this easy problem the heavy ball helps (on the Distill page it could also overshoot)
</details>

## 💻 Code break 5 - The learning rate: too small, about right, too large
Knob ④ - the most important one. Plain SGD, three learning rates.

In [ ]:
LR_SMALL = 0.001   # 👈
LR_GOOD = 0.3      # 👈
LR_LARGE = 5.0     # 👈

model_1 = make_model()
loss_small = train(model_1, torch.optim.SGD(model_1.parameters(), lr=LR_SMALL))
model_2 = make_model()
loss_good = train(model_2, torch.optim.SGD(model_2.parameters(), lr=LR_GOOD))
model_3 = make_model()
loss_large = train(model_3, torch.optim.SGD(model_3.parameters(), lr=LR_LARGE))

plt.plot(loss_small, label="too small")
plt.plot(loss_good, label="about right")
plt.plot(loss_large, label="too large")
plt.xlabel("epoch"); plt.ylabel("training loss"); plt.ylim(0, 2); plt.legend(); plt.show()
print("final loss  small:", round(loss_small[-1], 2), "  good:", round(loss_good[-1], 2),
      "  large:", round(loss_large[-1], 2))

🔧 **Change it:**
1. `LR_LARGE = 30.0` - what does the loss become?
2. `LR_LARGE = 1.0` - still too large?

<details><summary>👀 What you should see</summary>

- too small (0.001): the loss hardly moves (**1.10** after 30 epochs)
- about right (0.3): **0.06**
- too large (5.0): the curve jumps up and down and ends at **0.47**
- 30: the loss becomes **nan** ("not a number") - the training is broken
- 1.0 still works well here (**0.05**)
</details>

## 💻 Code break 6 (optional) - A learning-rate schedule
Knob ⑤: we only watch the learning rate - big steps first, small steps later.

In [ ]:
EPOCHS = 30        # 👈

model = make_model()
opt = torch.optim.SGD(model.parameters(), lr=0.1)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

learning_rates = []
for epoch in range(EPOCHS):
    learning_rates.append(sched.get_last_lr()[0])   # the learning rate in this epoch
    opt.step()                                      # (here the training of one epoch would happen)
    sched.step()                                    # once per epoch

plt.plot(learning_rates)
plt.xlabel("epoch"); plt.ylabel("learning rate"); plt.show()

🔧 **Change it:**
1. Replace `CosineAnnealingLR(opt, T_max=EPOCHS)` by `StepLR(opt, step_size=10, gamma=0.1)`. What shape?

<details><summary>👀 What you should see</summary>

- cosine: a smooth curve from 0.1 down to almost 0
- StepLR: stairs - 0.1, then 0.01 after 10 epochs, then 0.001 after 20
</details>